<a href="https://colab.research.google.com/github/jamshidbekmukhammedov/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Menu Detector!")

In [ ]:
#=============================
# Import Libraries           =
#=============================
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [ ]:
drive.mount('/content/drive')

In [ ]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print("Dataset_path:", DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping | class consolidation
    'cheesecake': 'dessert',     # label grouping | class consolidation
    'kebab': 'kebab',
    'pilaf': 'pilaf'
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print("NUM_CLASSES:", NUM_CLASSES)
print("Class_to_idx:", CLASS_TO_IDX)


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# =========================
# Custom Dataset Class    =
# =========================

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        # print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        # print('image_path', img_path)
        label = self.labels[idx]
        # print('label', label)
        try:
            image = Image.open(img_path)
            if image.mode == "P" or image.mode == "RGBA":  # png | gif | RGBA
                image = image.convert("RGBA").convert("RGB")
            else:
                image = image.convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# =========================
# Gather and Split Data   =
# =========================

all_images = []

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)
    print('class_path:', class_path)

    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)

split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

In [ ]:
tran_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [ ]:
train_loader = DataLoader(tran_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
# ====================
# Pretrained Model   =
# ====================

model = mobilenet_v2(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning | backbone
#

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss() # Loss Function
optimizer = optim.Adam(model.parameters(), lr=0.001)
torch.backends.cudnn.benchmark = True # Benchmark setting | Trick | 10%-20%

In [ ]:
# ==================
# Training Loop    =
# ==================

NUM_EPOCHS = 10
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
    model.train() # train mode
    running_loss = 0.0 # 70% | 30%Loss | 100%
    for images, labels in train_loader: # Forward and Backward(Backpropagation)
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad() # zero the gradient
        outputs = model(images) # Forward Pass | Dog | 5 Classes
        loss = criterion(outputs, labels) # Calculate Loss
        loss.backward()
        optimizer.step() # Adam optimizer
        running_loss += loss.item() # Track Loss

  # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total # Calculate Validation Accuracy
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {val_acc:.2f}%")

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), '/content/menu_detector.pth')
        print("Saved new best model!")

image_pathimage_path  /content/drive/MyDrive/food101_dataset/hamburger/172005.jpg/content/drive/MyDrive/food101_dataset/cheesecake/3199981.jpg

labellabel  0
2
images_length 3259
images_length 3259
image_pathimage_path  /content/drive/MyDrive/food101_dataset/chocolate_cake/1504717.jpg/content/drive/MyDrive/food101_dataset/hamburger/2961000.jpg

labellabel 2 0

image_path /content/drive/MyDrive/food101_dataset/hamburger/2157483.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/cheesecake/2292597.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/1422870.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/hot_dog/1265953.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/hot_dog/106222.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/2346909.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/1847512.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/443543.jp

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Streaming output truncated to the last 5000 lines.
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/2390259.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/3471329.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/hot_dog/1154965.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/hamburger/450282.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/hamburger/2537940.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/cheesecake/818687.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/hamburger/2686914.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/hot_dog/146834.jpg
label 1
image_path /content/drive/MyDrive/food101_dataset/hamburger/2062556.jpg
label 0
image_path /content/drive/MyDrive/food101_dataset/chocolate_cake/537929.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/cheesecake/1383955.jpg
label 2
image_path /content/drive/MyDrive/food101_dataset/hamburger/